<a href="https://colab.research.google.com/github/ggmirandac/HybridKinetics-IIQ3733/blob/master/Matrizanalitica.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [46]:
import numpy as np
import scipy
import matplotlib.pyplot as plt
import sympy
!pip install casadi
import casadi

In [47]:
ALL_PARAMS = [
    # PTS
    "kI_pyr_1",
    "kA_pep_1",
    "v_max_1",
    "Ka1_1",
    "Ka2_1",
    "Ka3_1",
    "n_g6p_1",
    "K_g6p_1",
    # PGI
    "kI_pep_2",
    "Km_g6p_2",
    "Km_f6p_2",
    "kcat_f_2",
    "kcat_r_2",
    # PFK
    "kI_f6p_3",
    "kI_fbp_3",
    "kI_gtp_3",
    "kI_pep_3",
    "kI_pi_3",
    "kA_adp_3",
    "kA_gdp_3",
    "Km_f6p_3",
    "Km_atp_3",
    "Km_fbp_3",
    "Km_adp_3",
    "kcat_f_3",
    "kcat_r_3",
    # FBA
    "kI_3pg_4",
    "kI_dhap_4",
    "kI_g3p_4",
    "kA_pep_4",
    "kcat_f_4",
    "kcat_r_4",
    "Km_fbp_4",
    "Km_g3p_4",
    "Km_dhap_4",
    # TPI
    "kcat_f_5",
    "kcat_r_5",
    "Km_dhap_5",
    "Km_g3p_5",
    # GAPDH
    "kI_adp_6",
    "kI_amp_6",
    "kI_atp_6",
    "kcat_f_6",
    "kcat_r_6",
    "Km_g3p_6",
    "Km_pi_6",
    "Km_nad_6",
    "Km_pgp_6",
    "Km_nadh_6",
    # PGK
    "kA_3pg_7",
    "kA_atp_7",
    "kcat_f_7",
    "kcat_r_7",
    "Km_pgp_7",
    "Km_adp_7",
    "Km_3pg_7",
    "Km_atp_7",
    # GPM
    "kI_pi_8",
    "kcat_f_8",
    "kcat_r_8",
    "Km_3pg_8",
    "Km_2pg_8",
    # ENO
    "kcat_f_9",
    "kcat_r_9",
    "Km_2pg_9",
    "Km_pep_9",
]
PARAM_RXN_MAP = {
    "pts": [
        "kI_pyr_1",
        "kA_pep_1",
        "v_max_1",
        "Ka1_1",
        "Ka2_1",
        "Ka3_1",
        "n_g6p_1",
        "K_g6p_1",
    ],
    "pgi": ["kI_pep_2", "Km_g6p_2", "Km_f6p_2", "kcat_f_2", "kcat_r_2"],
    "pfk": [
        "kI_f6p_3",
        "kI_fbp_3",
        "kI_gtp_3",
        "kI_pep_3",
        "kI_pi_3",
        "kA_adp_3",
        "kA_gdp_3",
        "Km_f6p_3",
        "Km_atp_3",
        "Km_fbp_3",
        "Km_adp_3",
        "kcat_f_3",
        "kcat_r_3",
    ],
    "fba": [
        "kI_3pg_4",
        "kI_dhap_4",
        "kI_g3p_4",
        "kA_pep_4",
        "kcat_f_4",
        "kcat_r_4",
        "Km_fbp_4",
        "Km_g3p_4",
        "Km_dhap_4",
    ],
    "tpi": ["kcat_f_5", "kcat_r_5", "Km_dhap_5", "Km_g3p_5"],
    "gap": [
        "kI_adp_6",
        "kI_amp_6",
        "kI_atp_6",
        "kcat_f_6",
        "kcat_r_6",
        "Km_g3p_6",
        "Km_pi_6",
        "Km_nad_6",
        "Km_pgp_6",
        "Km_nadh_6",
    ],
    "pgk": [
        "kA_3pg_7",
        "kA_atp_7",
        "kcat_f_7",
        "kcat_r_7",
        "Km_pgp_7",
        "Km_adp_7",
        "Km_3pg_7",
        "Km_atp_7",
    ],
    "gpm": ["kI_pi_8", "kcat_f_8", "kcat_r_8", "Km_3pg_8", "Km_2pg_8"],
    "eno": ["kcat_f_9", "kcat_r_9", "Km_2pg_9", "Km_pep_9"],
}

SUBSTRATE_MAP = {
    "D-Glucose-6-phosphate": "g6p",
    "D-Glucose 6-phosphate": "g6p",
    "ATP": "atp",
    "D-fructose 6-phosphate": "f6p",
    "fructose 6-phosphate": "f6p",
    "D-fructose 1,6-bisphosphate": "fbp",
    "D-glyceraldehyde 3-phosphate": "g3p",
    "NAD+": "nad",
    "phosphate": "pi",
}
SUBSTRATE_MAP = {
    k.lower().replace(" ", "_").replace("-", "_"): v for k, v in SUBSTRATE_MAP.items()
}
REACTION_MAP = {
    "1": "pts",
    "2": "pgi",
    "3": "pfk",
    "4": "fba",
    "5": "tpi",
    "6": "gap",
    "7": "pgk",
    "8": "gpm",
    "9": "eno",
}
REVERSE_REACTION_MAP = {v: k for k, v in REACTION_MAP.items()}


class EcoliCarbonKinetics:
    """
    Mechanistic kinetic model of E. coli central carbon metabolism.

    Each method computes the reaction flux for one glycolytic enzyme.
    All methods accept (constants, C, e) dicts so the call site in
    _ode_system is uniform across all 9 reactions.

    Attributes
    ----------
    metabolites : dict
        Metabolite concentrations keyed as 'C_<name>' (e.g. 'C_pep').
    enzymes : dict
        Enzyme concentrations keyed by gene name (e.g. 'PfkB', 'GapA').
    constants : dict
        Kinetic constants (Km, kcat, kI, kA, etc.).
    stoichiometric_matrix : pd.DataFrame
        9x9 stoichiometric matrix N (metabolites x reactions).
    """

    def __init__(self, metabolites: dict, enzymes: dict, constants: dict):
        self.metabolites = metabolites
        self.enzymes = enzymes
        self.constants = constants
        # Placeholder for the stoichiometric matrix construction.
        # This is needed because the __init__ method calls it, but its implementation
        # is not provided in the original code. For symbolic differentiation,
        # we don't strictly need a fully defined stoichiometric matrix.
        self.stoichiometric_matrix = self._construct_stoichiometric_matrix()

    def _construct_stoichiometric_matrix(self):
        # Placeholder implementation to avoid error
        return None

    def pts(self, constants: dict, C: dict, e: dict) -> float:
        """
        Phosphotransferase system (PTS).

        Reaction : glc + pep -> g6p + pyr
        Enzymes : PtsI, PtsH

        Rate is the product of three multiplicative terms:
            v = inhibition(pyr) * activation(pep) * kinetic(glc, g6p, pep/pyr)

        constants keys : kI_pyr_1, kA_pep_1, v_max_1, Ka1_1, Ka2_1, Ka3_1, n_g6p_1, K_g6p_1
        C keys : C_pyr, C_pep, C_glc, C_g6p
        e keys : (none)
        """
        kI_pyr_1 = constants["kI_pyr_1"]
        kA_pep_1 = constants["kA_pep_1"]
        v_max_1 = constants["v_max_1"]
        Ka1_1 = constants["Ka1_1"]
        Ka2_1 = constants["Ka2_1"]
        Ka3_1 = constants["Ka3_1"]
        n_g6p_1 = constants["n_g6p_1"]
        K_g6p_1 = constants["K_g6p_1"]

        C_pyr = C["C_pyr"]
        C_pep = C["C_pep"]
        C_glc = C["C_glc"]

        inhibition = kI_pyr_1 / (kI_pyr_1 + C_pyr)
        activation = C_pep / (C_pep + kA_pep_1)

        C_g6p = C["C_g6p"]
        num = v_max_1 * C_glc * (C_pep / C_pyr)
        den1 = Ka1_1 + Ka2_1 * (C_pep / C_pyr) + Ka3_1 * C_glc + C_glc * (C_pep / C_pyr)
        den2 = 1 + (C_g6p**n_g6p_1) / K_g6p_1
        kinetic = num / (den1 * den2)

        return inhibition * activation * kinetic

    def pgi(self, constants: dict, C: dict, e: dict) -> float:
        """
        Glucose-6-phosphate isomerase (PGI).
        EC: EC:5.3.1.9

        Reaction : g6p <=> f6p
        Enzyme : Pgi
        Inhibited by PEP.

        constants keys : kI_pep_2, Km_g6p_2, Km_f6p_2, kcat_f_2, kcat_r_2
        C keys : C_g6p, C_f6p, C_pep
        e keys : Pgi
        """
        kI_pep_2 = constants["kI_pep_2"]
        Km_g6p_2 = constants["Km_g6p_2"]
        Km_f6p_2 = constants["Km_f6p_2"]
        kcat_f_2 = constants["kcat_f_2"]
        kcat_r_2 = constants["kcat_r_2"]

        C_g6p = C["C_g6p"]
        C_f6p = C["C_f6p"]
        C_pep = C["C_pep"]

        h = kI_pep_2 / (kI_pep_2 + C_pep)
        num = kcat_f_2 * (C_g6p / Km_g6p_2) - kcat_r_2 * (C_f6p / Km_f6p_2)
        den = (C_g6p / Km_g6p_2 + 1) + (C_f6p / Km_f6p_2 + 1) - 1

        return e["Pgi"] * h * num / den

    def pfk(self, constants: dict, C: dict, e: dict) -> float:
        """
        Phosphofructokinase B (PFK).
        EC:2.7.1.11
        Reaction : f6p + ATP <=> fbp + ADP
        Enzyme : PfkB
        Inhibited by f6p, fbp, GTP, PEP, Pi.
        Activated by ADP, GDP.

        constants keys : kI_f6p_3, kI_fbp_3, kI_gtp_3, kI_pep_3, kI_pi_3,
                         kA_adp_3, kA_gdp_3,
                         Km_f6p_3, Km_atp_3, Km_fbp_3, Km_adp_3,
                         kcat_f_3, kcat_r_3
        C keys : C_f6p, C_atp, C_fbp, C_adp, C_gtp, C_gdp, C_pep, C_pi
        e keys : PfkB
        """
        kI_f6p_3 = constants["kI_f6p_3"]
        kI_fbp_3 = constants["kI_fbp_3"]
        kI_gtp_3 = constants["kI_gtp_3"]
        kI_pep_3 = constants["kI_pep_3"]
        kI_pi_3 = constants["kI_pi_3"]
        kA_adp_3 = constants["kA_adp_3"]
        kA_gdp_3 = constants["kA_gdp_3"]
        Km_f6p_3 = constants["Km_f6p_3"]
        Km_atp_3 = constants["Km_atp_3"]
        Km_fbp_3 = constants["Km_fbp_3"]
        Km_adp_3 = constants["Km_adp_3"]
        kcat_f_3 = constants["kcat_f_3"]
        kcat_r_3 = constants["kcat_r_3"]

        C_f6p = C["C_f6p"]
        C_atp = C["C_atp"]
        C_fbp = C["C_fbp"]
        C_adp = C["C_adp"]
        C_gtp = C["C_gtp"]
        C_gdp = C["C_gdp"]
        C_pep = C["C_pep"]
        C_pi = C["C_pi"]

        h = 1
        for conc, kI in [
            (C_f6p, kI_f6p_3),
            (C_fbp, kI_fbp_3),
            (C_gtp, kI_gtp_3),
            (C_pep, kI_pep_3),
            (C_pi, kI_pi_3),
        ]:
            h *= kI / (kI + conc)
        for conc, kA in [(C_adp, kA_adp_3), (C_gdp, kA_gdp_3)]:
            h *= conc / (kA + conc)

        num = kcat_f_3 * (C_f6p / Km_f6p_3) * (C_atp / Km_atp_3) - kcat_r_3 * (
            C_fbp / Km_fbp_3
        ) * (C_adp / Km_adp_3)
        den = (
            (C_f6p / Km_f6p_3 + 1) * (C_atp / Km_atp_3 + 1)
            + (C_fbp / Km_fbp_3 + 1) * (C_adp / Km_adp_3 + 1)
            - 1
        )

        return e["PfkB"] * h * num / den

    def fba(self, constants: dict, C: dict, e: dict) -> float:
        """
        Fructose-bisphosphate aldolase A (FBA).
        EC: 4.1.2.13
        Reaction : fbp <=> g3p + dhap
        Enzyme : FbaA
        Inhibited by 3PG, DHAP, G3P.
        Activated by PEP.

        constants keys : kI_3pg_4, kI_dhap_4, kI_g3p_4, kA_pep_4,
                         kcat_f_4, kcat_r_4, Km_fbp_4, Km_g3p_4, Km_dhap_4
        C keys : C_3pg, C_dhap, C_g3p, C_pep, C_fbp
        e keys : FbaA
        """
        kI_3pg_4 = constants["kI_3pg_4"]
        kI_dhap_4 = constants["kI_dhap_4"]
        kI_g3p_4 = constants["kI_g3p_4"]
        kA_pep_4 = constants["kA_pep_4"]
        kcat_f_4 = constants["kcat_f_4"]
        kcat_r_4 = constants["kcat_r_4"]
        Km_fbp_4 = constants["Km_fbp_4"]
        Km_g3p_4 = constants["Km_g3p_4"]
        Km_dhap_4 = constants["Km_dhap_4"]

        C_3pg = C["C_3pg"]
        C_dhap = C["C_dhap"]
        C_g3p = C["C_g3p"]
        C_pep = C["C_pep"]
        C_fbp = C["C_fbp"]

        h = 1
        for conc, kI in [(C_3pg, kI_3pg_4), (C_dhap, kI_dhap_4), (C_g3p, kI_g3p_4)]:
            h *= kI / (kI + conc)
        h *= C_pep / (kA_pep_4 + C_pep)

        num = kcat_f_4 * (C_fbp / Km_fbp_4) - kcat_r_4 * (C_g3p / Km_g3p_4) * (
            C_dhap / Km_dhap_4
        )
        den = (
            (C_fbp / Km_fbp_4 + 1)
            + (C_g3p / Km_g3p_4 + 1) * (C_dhap / Km_dhap_4 + 1)
            - 1
        )

        return e["FbaA"] * h * num / den

    def tpi(self, constants: dict, C: dict, e: dict) -> float:
        """
        Triosephosphate isomerase (TPI).
        EC: 5.3.1.1
        Reaction : dhap <=> g3p
        Enzyme : TpiA
        No allosteric regulation.

        constants keys : kcat_f_5, kcat_r_5, Km_dhap_5, Km_g3p_5
        C keys : C_dhap, C_g3p
        e keys : TpiA
        """
        kcat_f_5 = constants["kcat_f_5"]
        kcat_r_5 = constants["kcat_r_5"]
        Km_dhap_5 = constants["Km_dhap_5"]
        Km_g3p_5 = constants["Km_g3p_5"]

        C_dhap = C["C_dhap"]
        C_g3p = C["C_g3p"]

        num = kcat_f_5 * (C_dhap / Km_dhap_5) - kcat_r_5 * (C_g3p / Km_g3p_5)
        den = (C_dhap / Km_dhap_5 + 1) + (C_g3p / Km_g3p_5 + 1) - 1

        return e["TpiA"] * num / den

    def gap(self, constants: dict, C: dict, e: dict) -> float:
        """
        Glyceraldehyde-3-phosphate dehydrogenase A (GAP).
        EC: 1.2.1.12
        Reaction : g3p + Pi + NAD+ <=> pgp + NADH
        Enzyme : GapA
        Inhibited by ADP, AMP, ATP.

        constants keys : kI_adp_6, kI_amp_6, kI_atp_6,
                         kcat_f_6, kcat_r_6,
                         Km_g3p_6, Km_pi_6, Km_nad_6, Km_pgp_6, Km_nadh_6
        C keys : C_adp, C_amp, C_atp, C_g3p, C_pi, C_nad, C_pgp, C_nadh
        e keys : GapA
        """
        kI_adp_6 = constants["kI_adp_6"]
        kI_amp_6 = constants["kI_amp_6"]
        kI_atp_6 = constants["kI_atp_6"]
        kcat_f_6 = constants["kcat_f_6"]
        kcat_r_6 = constants["kcat_r_6"]
        Km_g3p_6 = constants["Km_g3p_6"]
        Km_pi_6 = constants["Km_pi_6"]
        Km_nad_6 = constants["Km_nad_6"]
        Km_pgp_6 = constants["Km_pgp_6"]
        Km_nadh_6 = constants["Km_nadh_6"]

        C_adp = C["C_adp"]
        C_amp = C["C_amp"]
        C_atp = C["C_atp"]
        C_g3p = C["C_g3p"]
        C_pi = C["C_pi"]
        C_nad = C["C_nad"]
        C_pgp = C["C_pgp"]
        C_nadh = C["C_nadh"]

        h = 1
        for conc, kI in [(C_adp, kI_adp_6), (C_amp, kI_amp_6), (C_atp, kI_atp_6)]:
            h *= kI / (kI + conc)

        num = kcat_f_6 * (C_g3p / Km_g3p_6) * (C_pi / Km_pi_6) * (
            C_nad / Km_nad_6
        ) - kcat_r_6 * (C_pgp / Km_pgp_6) * (C_nadh / Km_nadh_6)
        den = (
            (C_g3p / Km_g3p_6 + 1) * (C_pi / Km_pi_6 + 1) * (C_nad / Km_nad_6 + 1)
            + (C_pgp / Km_pgp_6 + 1) * (C_nadh / Km_nadh_6 + 1)
            - 1
        )

        return e["GapA"] * h * num / den

    def pgk(self, constants: dict, C: dict, e: dict) -> float:
        """
        Phosphoglycerate kinase (PGK).
        EC: 2.7.2.3
        Reaction : pgp + ADP <=> 3pg + ATP
        Enzyme : Pgk
        Activated by 3PG and ATP.

        constants keys : kA_3pg_7, kA_atp_7,
                         kcat_f_7, kcat_r_7,
                         Km_pgp_7, Km_adp_7, Km_3pg_7, Km_atp_7
        C keys : C_pgp, C_adp, C_3pg, C_atp
        e keys : Pgk
        """
        kA_3pg_7 = constants["kA_3pg_7"]
        kA_atp_7 = constants["kA_atp_7"]
        kcat_f_7 = constants["kcat_f_7"]
        kcat_r_7 = constants["kcat_r_7"]
        Km_pgp_7 = constants["Km_pgp_7"]
        Km_adp_7 = constants["Km_adp_7"]
        Km_3pg_7 = constants["Km_3pg_7"]
        Km_atp_7 = constants["Km_atp_7"]

        C_pgp = C["C_pgp"]
        C_adp = C["C_adp"]
        C_3pg = C["C_3pg"]
        C_atp = C["C_atp"]

        h = 1
        for conc, kA in [(C_3pg, kA_3pg_7), (C_atp, kA_atp_7)]:
            h *= conc / (kA + conc)

        num = kcat_f_7 * (C_pgp / Km_pgp_7) * (C_adp / Km_adp_7) - kcat_r_7 * (
            C_3pg / Km_3pg_7
        ) * (C_atp / Km_atp_7)
        den = (
            (C_pgp / Km_pgp_7 + 1) * (C_adp / Km_adp_7 + 1)
            + (C_3pg / Km_3pg_7 + 1) * (C_atp / Km_atp_7 + 1)
            - 1
        )

        return e["Pgk"] * h * num / den

    def gpm(self, constants: dict, C: dict, e: dict) -> float:
        """
        Phosphoglycerate mutase A (GPM).
        EC: 5.4.2.11
        Reaction : 3pg <=> 2pg
        Enzyme : GpmA
        Inhibited by Pi.

        constants keys : kI_pi_8, kcat_f_8, kcat_r_8, Km_3pg_8, Km_2pg_8
        C keys : C_3pg, C_2pg, C_pi
        e keys : GpmA
        """
        kI_pi_8 = constants["kI_pi_8"]
        kcat_f_8 = constants["kcat_f_8"]
        kcat_r_8 = constants["kcat_r_8"]
        Km_3pg_8 = constants["Km_3pg_8"]
        Km_2pg_8 = constants["Km_2pg_8"]

        C_3pg = C["C_3pg"]
        C_2pg = C["C_2pg"]
        C_pi = C["C_pi"]

        h = kI_pi_8 / (kI_pi_8 + C_pi)
        num = kcat_f_8 * (C_3pg / Km_3pg_8) - kcat_r_8 * (C_2pg / Km_2pg_8)
        den = (C_3pg / Km_3pg_8 + 1) + (C_2pg / Km_2pg_8 + 1) - 1

        return e["GpmA"] * h * num / den

    def eno(self, constants: dict, C: dict, e: dict) -> float:
        """
        Enolase (ENO).
        EC: 4.2.1.11
        Reaction : 2pg <=> pep
        Enzyme : Eno
        No allosteric regulation.

        constants keys : kcat_f_9, kcat_r_9, Km_2pg_9, Km_pep_9
        C keys : C_2pg, C_pep
        e keys : Eno
        """
        kcat_f_9 = constants["kcat_f_9"]
        kcat_r_9 = constants["kcat_r_9"]
        Km_2pg_9 = constants["Km_2pg_9"]
        Km_pep_9 = constants["Km_pep_9"]

        C_2pg = C["C_2pg"]
        C_pep = C["C_pep"]

        num = kcat_f_9 * (C_2pg / Km_2pg_9) - kcat_r_9 * (C_pep / Km_pep_9)
        den = (C_2pg / Km_2pg_9 + 1) + (C_pep / Km_pep_9 + 1) - 1

        return e["Eno"] * num / den

In [44]:
# Define symbolic variables for metabolites, enzymes, and constants
metabolite_names = [
    "pyr", "pep", "glc", "g6p", "f6p", "atp", "fbp", "adp", "gtp", "gdp",
    "pi", "3pg", "dhap", "g3p", "amp", "nad", "pgp", "nadh", "2pg"
]
C_sym = {f"C_{m}": sympy.Symbol(f"C_{m}") for m in metabolite_names}

enzyme_names = ["Pgi", "PfkB", "FbaA", "TpiA", "GapA", "Pgk", "GpmA", "Eno"]
e_sym = {enz: sympy.Symbol(enz) for enz in enzyme_names}

constants_sym = {p: sympy.Symbol(p) for p in ALL_PARAMS}

# Instantiate the EcoliCarbonKinetics class with symbolic variables
kinetic_model = EcoliCarbonKinetics(metabolites=C_sym, enzymes=e_sym, constants=constants_sym)

# Get symbolic expressions for each reaction rate
reaction_fluxes = {
    "pts": kinetic_model.pts(constants_sym, C_sym, e_sym),
    "pgi": kinetic_model.pgi(constants_sym, C_sym, e_sym),
    "pfk": kinetic_model.pfk(constants_sym, C_sym, e_sym),
    "fba": kinetic_model.fba(constants_sym, C_sym, e_sym),
    "tpi": kinetic_model.tpi(constants_sym, C_sym, e_sym),
    "gap": kinetic_model.gap(constants_sym, C_sym, e_sym),
    "pgk": kinetic_model.pgk(constants_sym, C_sym, e_sym),
    "gpm": kinetic_model.gpm(constants_sym, C_sym, e_sym),
    "eno": kinetic_model.eno(constants_sym, C_sym, e_sym),
}

# Define the ordered list of variables for differentiation (now parameters)
diff_variables = [constants_sym[p] for p in ALL_PARAMS]

# Create a list of reaction flux expressions in a consistent order
ordered_fluxes = [reaction_fluxes[REACTION_MAP[str(i)]] for i in range(1, 10)]

# Compute the Jacobian matrix
jacobian_matrix = sympy.Matrix(ordered_fluxes).jacobian(diff_variables)

print("Symbolic Jacobian Matrix (rows = reaction fluxes, columns = kinetic parameters):")
sympy.pprint(jacobian_matrix)


Se han truncado las últimas 5000 líneas del flujo de salida.
↪                                                                              ↪
↪                                                                              ↪
↪                                                                              ↪
↪                                                                              ↪
↪                                                                              ↪
↪                                                                              ↪
↪                                                                              ↪
↪                                                                              ↪
↪                                                                              ↪
↪                                                                              ↪
↪                                                                              ↪
↪                                               

In [48]:
np.shape(jacobian_matrix)

(9, 66)

In [45]:
# para ver si funcionaimprimimos la segunda columna de la matriz que debe tener 8 ceros
#de los parametros de la primera ecuación y luego 5 columnas con derivadas y luego ceros
sympy.pprint(jacobian_matrix)

⎡                                                     2                        ↪
⎢                                           C_glc⋅Cₚₑₚ ⋅kI_pyr_1⋅vₘₐₓ ₁        ↪
⎢- ─────────────────────────────────────────────────────────────────────────── ↪
⎢                                             ⎛     n_g6p_1    ⎞               ↪
⎢                                           2 ⎜C_g6p           ⎟ ⎛C_glc⋅Cₚₑₚ   ↪
⎢  C_pyr⋅(Cₚₑₚ + kAₚₑₚ ₁)⋅(C_pyr + kI_pyr_1) ⋅⎜──────────── + 1⎟⋅⎜────────── + ↪
⎢                                             ⎝  K_g6p_1       ⎠ ⎝  C_pyr      ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                           